<a href="https://colab.research.google.com/github/savinduharith-SDL/YOLO-for-HSI/blob/main/YOLO26_from_scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Imports

In [108]:
import torch.nn as nn
import torch
import copy, math

# Utils

In [109]:
def autopad(k, p=None, d=1):  # kernel, padding, dilation
    """Pad to 'same' shape outputs."""
    if d > 1:
        k = d * (k - 1) + 1 if isinstance(k, int) else [d * (x - 1) + 1 for x in k]  # actual kernel-size
    if p is None:
        p = k // 2 if isinstance(k, int) else [x // 2 for x in k]  # auto-pad
    return p

default_activation = nn.SiLU()
identity_activation = nn.Identity()

def dist2bbox(distance, anchor_points, xywh=True, dim=-1):
    """Transform distance(ltrb) to box(xywh or xyxy)."""
    lt, rb = distance.chunk(2, dim)
    x1y1 = anchor_points - lt
    x2y2 = anchor_points + rb
    if xywh:
        c_xy = (x1y1 + x2y2) / 2
        wh = x2y2 - x1y1
        return torch.cat([c_xy, wh], dim)  # xywh bbox
    return torch.cat((x1y1, x2y2), dim)  # xyxy bbox

def make_anchors(feats, strides, grid_cell_offset=0.5):
    """Generate anchors from features."""
    anchor_points, stride_tensor = [], []
    assert feats is not None
    dtype, device = feats[0].dtype, feats[0].device
    for i in range(len(feats)):  # use len(feats) to avoid TracerWarning from iterating over strides tensor
        stride = strides[i]
        h, w = feats[i].shape[2:] if isinstance(feats, list) else (int(feats[i][0]), int(feats[i][1]))
        sx = torch.arange(end=w, device=device, dtype=dtype) + grid_cell_offset  # shift x
        sy = torch.arange(end=h, device=device, dtype=dtype) + grid_cell_offset  # shift y
        sy, sx = torch.meshgrid(sy, sx)
        anchor_points.append(torch.stack((sx, sy), -1).view(-1, 2))
        stride_tensor.append(torch.full((h * w, 1), stride, dtype=dtype, device=device))
    return torch.cat(anchor_points), torch.cat(stride_tensor)

def make_divisible(x: int, divisor):
    if isinstance(divisor, torch.Tensor):
        divisor = int(divisor.max())  # to int
    return math.ceil(x / divisor) * divisor

def initialize_weights(model):
    """Initialize model weights, biases, and module settings to default values."""
    for m in model.modules():
        t = type(m)
        if t is nn.Conv2d:
            pass  # nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
        elif t is nn.BatchNorm2d:
            m.eps = 1e-3
            m.momentum = 0.03
        elif t in {nn.Hardswish, nn.LeakyReLU, nn.ReLU, nn.ReLU6, nn.SiLU}:
            m.inplace = True

# Modules

Conv Block

In [110]:
class Conv(nn.Module):

    default_act = nn.SiLU()  # default activation

    def __init__(self, c1, c2, k=1, s=1, p=None, g=1, d=1, act=True):
        super().__init__()
        self.conv = nn.Conv2d(c1, c2, k, s, autopad(k, p, d), groups=g, dilation=d, bias=False)
        self.bn = nn.BatchNorm2d(c2)
        self.act = self.default_act if act is True else act if isinstance(act, nn.Module) else nn.Identity()

    def forward(self, x):
        return self.act(self.bn(self.conv(x)))

    def forward_fuse(self, x):
        return self.act(self.conv(x))

DWConv Block

In [111]:
class DWConv(Conv):
    def __init__(self, c1, c2, k=1, s=1, d=1, act=True):
        super().__init__(c1, c2, k, s, g=math.gcd(c1, c2), d=d, act=act)

Bottleneck Block

In [112]:
class Bottleneck(nn.Module):
    def __init__(
        self, c1: int, c2: int, shortcut: bool = True, g: int = 1, k: tuple[int, int] = (3, 3), e: float = 0.5
    ):
        super().__init__()
        c_ = int(c2 * e)  # hidden channels
        self.cv1 = Conv(c1, c_, k[0], 1)
        self.cv2 = Conv(c_, c2, k[1], 1, g=g)
        self.add = shortcut and c1 == c2

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.cv2(self.cv1(x)) if self.add else self.cv2(self.cv1(x))

C3K Block

In [113]:
class C3(nn.Module):
    def __init__(self, c1: int, c2: int, n: int = 1, shortcut: bool = True, g: int = 1, e: float = 0.5):
        super().__init__()
        c_ = int(c2 * e)  # hidden channels
        self.cv1 = Conv(c1, c_, 1, 1)
        self.cv2 = Conv(c1, c_, 1, 1)
        self.cv3 = Conv(2 * c_, c2, 1)  # optional act=FReLU(c2)
        self.m = nn.Sequential(*(Bottleneck(c_, c_, shortcut, g, k=((1, 1), (3, 3)), e=1.0) for _ in range(n)))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.cv3(torch.cat((self.m(self.cv1(x)), self.cv2(x)), 1))

class C3k(C3):

    def __init__(self, c1: int, c2: int, n: int = 1, shortcut: bool = True, g: int = 1, e: float = 0.5, k: int = 3):
        super().__init__(c1, c2, n, shortcut, g, e)
        c_ = int(c2 * e)  # hidden channels
        # self.m = nn.Sequential(*(RepBottleneck(c_, c_, shortcut, g, k=(k, k), e=1.0) for _ in range(n)))
        self.m = nn.Sequential(*(Bottleneck(c_, c_, shortcut, g, k=(k, k), e=1.0) for _ in range(n)))

C2f Block

In [114]:
class C2f(nn.Module):

    def __init__(self, c1: int, c2: int, n: int = 1, shortcut: bool = False, g: int = 1, e: float = 0.5):
        super().__init__()
        self.c = int(c2 * e)  # hidden channels
        self.cv1 = Conv(c1, 2 * self.c, 1, 1)
        self.cv2 = Conv((2 + n) * self.c, c2, 1)  # optional act=FReLU(c2)
        self.m = nn.ModuleList(Bottleneck(self.c, self.c, shortcut, g, k=((3, 3), (3, 3)), e=1.0) for _ in range(n))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        y = list(self.cv1(x).chunk(2, 1))
        y.extend(m(y[-1]) for m in self.m)
        return self.cv2(torch.cat(y, 1))

    def forward_split(self, x: torch.Tensor) -> torch.Tensor:
        y = self.cv1(x).split((self.c, self.c), 1)
        y = [y[0], y[1]]
        y.extend(m(y[-1]) for m in self.m)
        return self.cv2(torch.cat(y, 1))

Attention Block

In [115]:
class Attention(nn.Module):

    def __init__(self, dim: int, num_heads: int = 8, attn_ratio: float = 0.5):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.key_dim = int(self.head_dim * attn_ratio)
        self.scale = self.key_dim**-0.5
        nh_kd = self.key_dim * num_heads
        h = dim + nh_kd * 2
        self.qkv = Conv(dim, h, 1, act=False)
        self.proj = Conv(dim, dim, 1, act=False)
        self.pe = Conv(dim, dim, 3, 1, g=dim, act=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, C, H, W = x.shape
        N = H * W
        qkv = self.qkv(x)
        q, k, v = qkv.view(B, self.num_heads, self.key_dim * 2 + self.head_dim, N).split(
            [self.key_dim, self.key_dim, self.head_dim], dim=2
        )

        attn = (q.transpose(-2, -1) @ k) * self.scale
        attn = attn.softmax(dim=-1)
        x = (v @ attn.transpose(-2, -1)).view(B, C, H, W) + self.pe(v.reshape(B, C, H, W))
        x = self.proj(x)
        return x

PSA Block

In [116]:
class PSABlock(nn.Module):

    def __init__(self, c: int, attn_ratio: float = 0.5, num_heads: int = 4, shortcut: bool = True) -> None:
        super().__init__()
        self.attn = Attention(c, attn_ratio=attn_ratio, num_heads=num_heads)
        self.ffn = nn.Sequential(Conv(c, c * 2, 1), Conv(c * 2, c, 1, act=False))
        self.add = shortcut

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(x) if self.add else self.attn(x)
        x = x + self.ffn(x) if self.add else self.ffn(x)
        return x

C3k2 Block

In [117]:
class C3k2(C2f):
    def __init__(
        self,
        c1: int,
        c2: int,
        n: int = 1,
        c3k: bool = False,
        e: float = 0.5,
        attn: bool = False,
        g: int = 1,
        shortcut: bool = True,
    ):
        super().__init__(c1, c2, n, shortcut, g, e)
        self.m = nn.ModuleList(
            nn.Sequential(
                Bottleneck(self.c, self.c, shortcut, g),
                PSABlock(self.c, attn_ratio=0.5, num_heads=max(self.c // 64, 1)),
            )
            if attn
            else C3k(self.c, self.c, 2, shortcut, g)
            if c3k
            else Bottleneck(self.c, self.c, shortcut, g)
            for _ in range(n)
        )

SPPF Block

In [118]:
class SPPF(nn.Module):
    def __init__(self, c1: int, c2: int, k: int = 5, n: int = 3, shortcut: bool = False):
        super().__init__()
        c_ = c1 // 2  # hidden channels
        self.cv1 = Conv(c1, c_, 1, 1, act=False)
        self.cv2 = Conv(c_ * (n + 1), c2, 1, 1)
        self.m = nn.MaxPool2d(kernel_size=k, stride=1, padding=k // 2)
        self.n = n
        self.add = shortcut and c1 == c2

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        y = [self.cv1(x)]
        y.extend(self.m(y[-1]) for _ in range(getattr(self, "n", 3)))
        y = self.cv2(torch.cat(y, 1))
        return y + x if getattr(self, "add", False) else y

C2PSA Block

In [119]:
class C2PSA(nn.Module):

    def __init__(self, c1: int, c2: int, n: int = 1, e: float = 0.5):
        super().__init__()
        assert c1 == c2
        self.c = int(c1 * e)
        self.cv1 = Conv(c1, 2 * self.c, 1, 1)
        self.cv2 = Conv(2 * self.c, c1, 1)

        self.m = nn.Sequential(*(PSABlock(self.c, attn_ratio=0.5, num_heads=self.c // 64) for _ in range(n)))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        a, b = self.cv1(x).split((self.c, self.c), dim=1)
        b = self.m(b)
        return self.cv2(torch.cat((a, b), 1))

Concat

In [146]:
class Concat(nn.Module):
    def __init__(self, dimension=1):
        super().__init__()
        self.d = dimension

    def forward(self, x: list[torch.Tensor]):
        return torch.cat(x, self.d)

Head

In [147]:
class Detect(nn.Module):
    """YOLO26 detect head: native end-to-end (NMS-free), no DFL."""
    dynamic = False
    export  = False
    shape   = None
    anchors = torch.empty(0)
    strides = torch.empty(0)
    max_det = 300

    def __init__(self, nc=80, reg_max=1, ch=()):
        super().__init__()
        self.nc = nc
        self.nl = len(ch)
        self.reg_max = reg_max                 # 1 for YOLO26 -> DFL removed
        self.no = nc + reg_max * 4
        self.stride = torch.zeros(self.nl)     # filled by dummy forward in the model

        c2 = max(16, ch[0] // 4, reg_max * 4)
        c3 = max(ch[0], min(nc, 100))

        # one2many head (rich supervision during training)
        self.cv2 = nn.ModuleList(
            nn.Sequential(Conv(x, c2, 3), Conv(c2, c2, 3), nn.Conv2d(c2, 4 * reg_max, 1)) for x in ch
        )
        self.cv3 = nn.ModuleList(
            nn.Sequential(
                nn.Sequential(DWConv(x, x, 3), Conv(x, c3, 1)),
                nn.Sequential(DWConv(c3, c3, 3), Conv(c3, c3, 1)),
                nn.Conv2d(c3, nc, 1),
            ) for x in ch
        )
        # one2one head (NMS-free inference). deepcopy -> identical init to one2many
        self.one2one_cv2 = copy.deepcopy(self.cv2)
        self.one2one_cv3 = copy.deepcopy(self.cv3)

        self.dfl = nn.Identity()   # Identity for YOLO26

    # ---- shared head application ----
    def _heads(self, x, box_head, cls_head):
        bs = x[0].shape[0]
        boxes  = torch.cat([box_head[i](x[i]).view(bs, 4 * self.reg_max, -1) for i in range(self.nl)], -1)
        scores = torch.cat([cls_head[i](x[i]).view(bs, self.nc, -1)         for i in range(self.nl)], -1)
        return {"boxes": boxes, "scores": scores, "feats": x}

    def forward(self, x):
        if self.training:
            one2many = self._heads(x, self.cv2, self.cv3)
            # one2one trains on DETACHED feats so its gradients don't disturb the shared body
            one2one  = self._heads([xi.detach() for xi in x], self.one2one_cv2, self.one2one_cv3)
            return {"one2many": one2many, "one2one": one2one}

        one2one = self._heads(x, self.one2one_cv2, self.one2one_cv3)   # inference uses one2one only
        y = self._inference(one2one)
        y = self.postprocess(y.permute(0, 2, 1))                       # top-k, NO NMS
        return y if self.export else (y, one2one)

    def _inference(self, p):
        shape = p["feats"][0].shape
        if self.dynamic or self.shape != shape:
            self.anchors, self.strides = (a.transpose(0, 1) for a in make_anchors(p["feats"], self.stride, 0.5))
            self.shape = shape
        dbox = self.decode_bboxes(self.dfl(p["boxes"]), self.anchors.unsqueeze(0)) * self.strides
        return torch.cat((dbox, p["scores"].sigmoid()), 1)

    def decode_bboxes(self, bboxes, anchors):
        # end2end decodes to xyxy (what postprocess consumes)
        return dist2bbox(bboxes, anchors, xywh=False, dim=1)

    # ---- NMS-free selection ----
    def postprocess(self, preds):
        # preds: (b, num_anchors, 4 + nc), boxes already xyxy
        boxes, scores = preds.split([4, self.nc], dim=-1)
        scores, conf, idx = self.get_topk_index(scores, self.max_det)
        boxes = boxes.gather(dim=1, index=idx.repeat(1, 1, 4))
        return torch.cat([boxes, scores, conf], dim=-1)   # (b, max_det, 6): x1y1x2y2, conf, cls

    def get_topk_index(self, scores, max_det):
        b, anchors, nc = scores.shape
        k = min(max_det, anchors)
        ori_index = scores.max(dim=-1)[0].topk(k)[1].unsqueeze(-1)
        scores = scores.gather(dim=1, index=ori_index.repeat(1, 1, nc))
        scores, index = scores.flatten(1).topk(k)
        idx = ori_index[torch.arange(b)[..., None], index // nc]
        return scores[..., None], (index % nc)[..., None].float(), idx

    def bias_init(self):
        for cv2, cv3 in ((self.cv2, self.cv3), (self.one2one_cv2, self.one2one_cv3)):
            for a, b, s in zip(cv2, cv3, self.stride):
                a[-1].bias.data[:] = 1.0
                b[-1].bias.data[: self.nc] = math.log(5 / self.nc / (640 / s) ** 2)

# YOLO26

In [162]:
SCALES = {
    #      depth  width  max_channels
    "n": (0.50, 0.25, 1024),
    "s": (0.50, 0.50, 1024),
    "m": (0.50, 1.00, 512),
    "l": (1.00, 1.00, 512),
    "x": (1.00, 1.50, 512),
}

class YOLO26(nn.Module):
    def __init__(self, scale="n", nc=80, ch=3):
        super().__init__()
        depth, width, max_ch = SCALES[scale]

        def w(c):                                   # scale + round a channel count
            return make_divisible(min(c, max_ch) * width, 8)
        def n(r):                                   # scale a repeat count
            return max(round(r * depth), 1) if r > 1 else r

        # ---- backbone ----
        self.b0  = Conv(ch,       w(64),   3, 2)                       # 0  P1/2
        self.b1  = Conv(w(64),    w(128),  3, 2)                       # 1  P2/4
        self.b2  = C3k2(w(128),   w(256),  n(2), False, e=0.25)        # 2
        self.b3  = Conv(w(256),   w(256),  3, 2)                       # 3  P3/8
        self.b4  = C3k2(w(256),   w(512),  n(2), False, e=0.25)        # 4  -> P3
        self.b5  = Conv(w(512),   w(512),  3, 2)                       # 5  P4/16
        self.b6  = C3k2(w(512),   w(512),  n(2), True)                 # 6  -> P4
        self.b7  = Conv(w(512),   w(1024), 3, 2)                       # 7  P5/32
        self.b8  = C3k2(w(1024),  w(1024), n(2), True)                 # 8
        self.b9  = SPPF(w(1024),  w(1024), 5, 3, True)                 # 9  shortcut=True
        self.b10 = C2PSA(w(1024), w(1024), n(2))                       # 10 -> P5

        # ---- head / neck (YOLO26: c3k=True throughout; last block n=1) ----
        self.up  = nn.Upsample(scale_factor=2, mode="nearest")
        self.cat = Concat(1)                                           # stateless, reused
        self.h13 = C3k2(w(1024) + w(512),  w(512),  n(2), True)        # 13
        self.h16 = C3k2(w(512)  + w(512),  w(256),  n(2), True)        # 16 -> P3 out
        self.h17 = Conv(w(256),  w(256),   3, 2)                       # 17
        self.h19 = C3k2(w(256)  + w(512),  w(512),  n(2), True)        # 19 -> P4 out
        self.h20 = Conv(w(512),  w(512),   3, 2)                       # 20
        self.h22 = C3k2(w(512) + w(1024), w(1024), 1, c3k=True, attn=True)           # 22 -> P5 out (n=1)

        self.detect = Detect(nc, reg_max=1, ch=(w(256), w(512), w(1024)))

        # ---- strides via dummy forward (eval so BN stats untouched), then biases & weights ----
        s = 256
        self.eval()
        with torch.no_grad():
            feats = self.forward_features(torch.zeros(1, ch, s, s))
        self.detect.stride = torch.tensor([s / f.shape[-2] for f in feats])  # -> [8,16,32]
        self.stride = self.detect.stride
        self.detect.bias_init()
        self.train()
        initialize_weights(self)

    def forward_features(self, x):
        x = self.b0(x); x = self.b1(x); x = self.b2(x); x = self.b3(x)
        p3 = self.b4(x)
        x = self.b5(p3)
        p4 = self.b6(x)
        x = self.b7(p4); x = self.b8(x); x = self.b9(x)
        p5 = self.b10(x)

        x   = self.cat([self.up(p5),   p4]);  h13 = self.h13(x)
        x   = self.cat([self.up(h13),  p3]);  h16 = self.h16(x)    # P3 out
        x   = self.cat([self.h17(h16), h13]); h19 = self.h19(x)    # P4 out
        x   = self.cat([self.h20(h19), p5]);  h22 = self.h22(x)    # P5 out
        return [h16, h19, h22]

    def forward(self, x):
        return self.detect(self.forward_features(x))

In [163]:
model = YOLO26("n", nc=80)
print(sum(p.numel() for p in model.parameters()))   # 2,572,280
print(model.stride)                                  # tensor([ 8., 16., 32.])

2572280
tensor([ 8., 16., 32.])
